# GeoSense – training our own models on Colab (Mohamed)

Runtime → Change runtime type → **T4 GPU**. Checkpoints go to Google Drive so a disconnect loses nothing: re-run the cells and add `--resume`.

In [ ]:
!nvidia-smi -L

## 1. Code + Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'mohamed/training'   # switch to 'main' once the pull request is merged
!rm -rf /content/geosense && git clone -q -b {BRANCH} https://github.com/arson-glitch/geosense.git /content/geosense
%cd /content/geosense
!pip install -q -e . kagglehub

# checkpoints live on Drive, so a disconnect loses nothing (re-run with --resume)
DRIVE = '/content/drive/MyDrive/geosense'
!mkdir -p {DRIVE}/checkpoints {DRIVE}/predictions {DRIVE}/results_runs
!rm -rf checkpoints && ln -s {DRIVE}/checkpoints checkpoints

## 2. Dataset (~7 GB, ~5 min)
If kagglehub asks for credentials: kaggle.com → Settings → API → Create token, then `kagglehub.login()`.

In [ ]:
import kagglehub
path = kagglehub.dataset_download('ubitquitin/geolocation-geoguessr-images-50k')
ROOT = f'{path}/compressed_dataset'
!ls {ROOT} | head -3 && ls {ROOT} | wc -l

## 3. Quick check – does everything run?
The first run also builds the image cache in `/content/cache` (~10 min, once per runtime): every later epoch is ~4x faster.

In [ ]:
!python -m geosense.train --root {ROOT} --model cnn_scratch --subset 2000 --epochs 2 --cache /content/cache

## 4. CNN from scratch
Don't over-tune it: one or two runs is enough. Early stopping ends it when val macro-F1 stops improving.

In [ ]:
!python -m geosense.train --root {ROOT} --model cnn_scratch --epochs 30 --workers 2 --cache /content/cache 2>&1 | tee -a {DRIVE}/results_runs/cnn_scratch_log.txt
# disconnected? run the same line again with --resume

## 5. ResNet-50 transfer learning
Stage 1: 3 epochs head only (lr 1e-3). Stage 2: layer4 unfrozen (lr 1e-4).

In [ ]:
!python -m geosense.train --root {ROOT} --model resnet50 --epochs 15 --stage1-epochs 3 --unfreeze-blocks 1 --workers 2 --cache /content/cache 2>&1 | tee -a {DRIVE}/results_runs/resnet50_log.txt

## 6. Save logits + curves to Drive, check validation scores
Lost the runtime after training? `--export-only` re-creates the logits from `best.pt` on Drive. Test logits: **only at the end of the project**, with `--export-only --export val,test`.

In [ ]:
!cp predictions/cnn_scratch_* predictions/resnet50_* {DRIVE}/predictions/ ; cp results/*_history.csv {DRIVE}/results_runs/ ; ls -la {DRIVE}/predictions {DRIVE}/results_runs
!python -c "from geosense.predictions import check_val; check_val()"   # val top-1 / top-5 / macro-F1, all models